# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

In [47]:
from pyspark.sql.functions import *

####  Run this cell to set up and start your interactive session.


In [1]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: 94cc4e0a-1508-41e2-aa0a-451726c73935
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 94cc4e0a-1508-41e2-aa0a-451726c73935 to get into ready status...
Session 94cc4e0a-1508-41e2-aa0a-451726c73935 

#### Example: Create a DynamicFrame from a table in the AWS Glue Data Catalog and display its schema


In [36]:
dyf = glueContext.create_dynamic_frame.from_catalog(database='glue_project', table_name='titles')
dyf.printSchema()

root
|-- duration_minutes: string
|-- duration_seasons: string
|-- type: string
|-- title: string
|-- date_added: string
|-- release_year: string
|-- rating: string
|-- description: string
|-- show_id: string


#### Example: Convert the DynamicFrame to a Spark DataFrame and display a sample of the data


In [37]:
df = dyf.toDF()
df.show()

+----------------+----------------+-------+--------------------+----------+------------+------+--------------------+--------+
|duration_minutes|duration_seasons|   type|               title|date_added|release_year|rating|         description| show_id|
+----------------+----------------+-------+--------------------+----------+------------+------+--------------------+--------+
|                |               1|TV Show|Welcome to the Fa...| 7/27/2018|        2018| TV-MA|When an evicted s...|80212251|
|             101|                |  Movie|American Circumci...|12/16/2018|        2017| TV-14|With interviews f...|81000861|
|              95|                |  Movie|           Lionheart|  1/4/2019|        2018| TV-PG|When her father f...|81030789|
|             107|                |  Movie|The Adventures of...|11/20/2019|        2011|    PG|This 3-D motion c...|70121502|
|             181|                |  Movie|Jis Desh Men Gang...|12/31/2019|        1960| TV-14|Falling in with a...|60

In [38]:
df.count()

6236


In [39]:
df = df.withColumn(
    "duration_minutes",
    when((col("duration_minutes").isNull()) | (col("duration_minutes") == ""), "0")
    .otherwise(col("duration_minutes"))
)

In [40]:
df = df.withColumn(
    "duration_seasons",
    when((col("duration_seasons").isNull()) | (col("duration_seasons") == ""), "1")
    .otherwise(col("duration_seasons"))
)

In [42]:
df = df.withColumn("duration_minutes", col("duration_minutes").cast("int"))\
       .withColumn("duration_seasons", col("duration_seasons").cast("int"))

In [43]:
df.printSchema()

root
 |-- duration_minutes: integer (nullable = true)
 |-- duration_seasons: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- description: string (nullable = true)
 |-- show_id: string (nullable = true)


In [48]:
from pyspark.sql.functions import col, split

In [51]:
df=df.withColumn("ShortTitle", split(col("title"),":")[0])


In [53]:
df=df.withColumn("rating",split(col("rating"),"-")[0])

In [56]:
df=df.withColumn("flag_id",when(col("type")=="Movie",1)\
              .when(col("type")=="TV Show",2)\
              .otherwise(0))

In [58]:
df.show()

+----------------+----------------+-------+--------------------+----------+------------+------+--------------------+--------+--------------------+-------+
|duration_minutes|duration_seasons|   type|               title|date_added|release_year|rating|         description| show_id|          ShortTitle|flag_id|
+----------------+----------------+-------+--------------------+----------+------------+------+--------------------+--------+--------------------+-------+
|               0|               1|TV Show|Welcome to the Fa...| 7/27/2018|        2018|    TV|When an evicted s...|80212251|Welcome to the Fa...|      2|
|             101|               1|  Movie|American Circumci...|12/16/2018|        2017|    TV|With interviews f...|81000861|American Circumci...|      1|
|              95|               1|  Movie|           Lionheart|  1/4/2019|        2018|    TV|When her father f...|81030789|           Lionheart|      1|
|             107|               1|  Movie|The Adventures of...|11/20/

In [59]:
from pyspark.sql.window import Window

In [60]:
df = df.withColumn("duration_ranking", dense_rank().over(Window.orderBy(col("duration_minutes").desc())))

In [69]:
df.groupBy(col("type")).agg(count("*").alias("total")).show()

+-------+-----+
|   type|total|
+-------+-----+
|TV Show| 1969|
|  Movie| 4265|
|       |    1|
|   1944|    1|
+-------+-----+


In [70]:
df.count()

6236


In [75]:
df=df.filter((col("type")=="TV Show") | (col("type")=="Movie"))

In [76]:
df.count()

6234


In [77]:
df.write.format("parquet")\
    .mode("append")\
    .save("s3://project-glue-uk/silver_data/titles_silver/")